In [3]:
#Import the libraries 
import pandas as pd
import numpy as np

In [13]:
#Load the dataset 
df1=pd.read_csv("building_materials_transactions.csv")
df2=pd.read_csv("macro_drivers_weekly.csv")

In [11]:
#Cleaning df1

In [12]:
# Standardize column names
df1.columns = (df1.columns.str.strip().str.lower()
               .str.replace(" ", "_").str.replace(r"[^\w]", "", regex=True))

print(df1.shape, df1.dtypes, sep="\n")
print("Missing %:\n", (df1.isna().mean() * 100).loc[lambda s: s > 0].sort_values(ascending=False))

# Dedupe, strip text
df1 = df1.drop_duplicates()
obj = df1.select_dtypes("object").columns
df1[obj] = df1[obj].apply(lambda s: s.str.strip())

# df1["date"] = pd.to_datetime(df1["date"], errors="coerce")  # uncomment per column
# df1["sales"] = pd.to_numeric(df1["sales"], errors="coerce")

# Fill missing: median for numeric, mode for text
num = df1.select_dtypes("number").columns
df1[num] = df1[num].fillna(df1[num].median())
df1[obj] = df1[obj].fillna(df1[obj].mode().iloc[0])

# IQR outlier counts (report only, not removed)
q1, q3 = df1[num].quantile(.25), df1[num].quantile(.75)
iqr = q3 - q1
print("Outliers:\n", ((df1[num] < q1 - 1.5*iqr) | (df1[num] > q3 + 1.5*iqr)).sum())

print("Missing left:", df1.isna().sum().sum(), "| Dupes:", df1.duplicated().sum(), "| Shape:", df1.shape)
df1.to_csv("cleaned_data.csv", index=False)

(340000, 15)
date                     object
year                      int64
month                     int64
week_of_year              int64
region                   object
product_category         object
sku                      object
channel                  object
customer_type            object
units                     int64
unit_price              float64
revenue                 float64
housing_starts_index    float64
lumber_price_index      float64
mortgage_rate           float64
dtype: object
Missing %:
 Series([], dtype: float64)
Outliers:
 year                        0
month                       0
week_of_year                0
units                   33368
unit_price              57287
revenue                 23368
housing_starts_index        0
lumber_price_index      57262
mortgage_rate               0
dtype: int64
Missing left: 0 | Dupes: 0 | Shape: (340000, 15)


In [ ]:
#Cleaning df2

In [22]:
import pandas as pd

# ==========================================
# 1. Load the dataset
# ==========================================

df2 = pd.read_csv("macro_drivers_weekly.csv")

# ==========================================
# 2. Standardize column names
# ==========================================

df2.columns = (
    df2.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[^\w]", "", regex=True)
)

print("Shape:", df2.shape)
print("\nColumns:")
print(df2.columns.tolist())

print("\nData Types:")
print(df2.dtypes)

# ==========================================
# 3. Check missing values
# ==========================================

print("\nMissing Values:")
print(df2.isna().sum())

print("\nMissing Percentage:")
print(
    (df2.isna().mean() * 100)
    .round(2)
    .sort_values(ascending=False)
)

# ==========================================
# 4. Remove duplicate rows
# ==========================================

df2 = df2.drop_duplicates()

# ==========================================
# 5. Strip whitespace from text columns
# ==========================================

obj = df2.select_dtypes(include="object").columns

for col in obj:
    df2[col] = df2[col].str.strip()

# ==========================================
# 6. Convert week to datetime
# ==========================================

if "week" in df2.columns:
    df2["week"] = pd.to_datetime(
        df2["week"],
        errors="coerce"
    )

# ==========================================
# 7. Convert numeric columns to numeric
# ==========================================

numeric_columns = [
    "housing_starts_index",
    "lumber_price_index",
    "mortgage_rate",
    "season_factor"
]

for col in numeric_columns:
    if col in df2.columns:
        df2[col] = pd.to_numeric(
            df2[col],
            errors="coerce"
        )

# ==========================================
# 8. Identify numeric columns
# ==========================================

num = df2.select_dtypes(include="number").columns

# ==========================================
# 9. Fill missing numeric values
#    Using median
# ==========================================

for col in num:
    df2[col] = df2[col].fillna(
        df2[col].median()
    )

# ==========================================
# 10. Re-identify object columns
# ==========================================

obj = df2.select_dtypes(include="object").columns

# ==========================================
# 11. Fill missing text values
#     Using mode
# ==========================================

for col in obj:
    if df2[col].isna().any():
        df2[col] = df2[col].fillna(
            df2[col].mode()[0]
        )

# ==========================================
# 12. IQR Outlier Detection
#     Outliers are reported, NOT removed
# ==========================================

q1 = df2[num].quantile(0.25)
q3 = df2[num].quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - (1.5 * iqr)
upper_bound = q3 + (1.5 * iqr)

outliers = (
    (df2[num] < lower_bound) |
    (df2[num] > upper_bound)
).sum()

print("\nOutlier Counts:")
print(outliers)

# ==========================================
# 13. Final data quality check
# ==========================================

print("\n========== FINAL DATA QUALITY CHECK ==========")

print(
    "Missing values:",
    df2.isna().sum().sum()
)

print(
    "Duplicate rows:",
    df2.duplicated().sum()
)

print(
    "Dataset shape:",
    df2.shape
)

print("\nFinal Data Types:")
print(df2.dtypes)

# ==========================================
# 14. Preview cleaned dataset
# ==========================================

print("\nCleaned Dataset:")
print(df2.head())

# ==========================================
# 15. Save cleaned dataset
# ==========================================

df2.to_csv(
    "macro_drivers_weekly_cleaned.csv",
    index=False
)

print("\nCleaned dataset saved as:")
print("macro_drivers_weekly_cleaned.csv")

Shape: (313, 5)

Columns:
['week', 'housing_starts_index', 'lumber_price_index', 'mortgage_rate', 'season_factor']

Data Types:
week                     object
housing_starts_index    float64
lumber_price_index      float64
mortgage_rate           float64
season_factor           float64
dtype: object

Missing Values:
week                    0
housing_starts_index    0
lumber_price_index      0
mortgage_rate           0
season_factor           0
dtype: int64

Missing Percentage:
week                    0.0
housing_starts_index    0.0
lumber_price_index      0.0
mortgage_rate           0.0
season_factor           0.0
dtype: float64

Outlier Counts:
housing_starts_index     0
lumber_price_index      56
mortgage_rate            0
season_factor            0
dtype: int64

========== FINAL DATA QUALITY CHECK ==========
Missing values: 0
Duplicate rows: 0
Dataset shape: (313, 5)

Final Data Types:
week                    datetime64[ns]
housing_starts_index           float64
lumber_price_index 